# 04 · Stretch: 4-bit Llama2-7B, NF4 DiaBlo vs QLoRA (optional)
**Important:** the paper's quantized results (Table 4) use **GPTQ/MagR** quantization and **Math10K** training. Neither the quantizers nor
Math10K processing are in the repo, so this notebook **cannot reproduce Table 4's numbers**. What it can test is the closely related claim
that, on the same 4-bit base, replacing LoRA with DiaBlo helps. We use bitsandbytes NF4 and GSM8k-train.
For reference, the paper reports GSM8k accuracy for 4-bit Llama2-7B trained on Math10K: QLoRA r=64 gets 42.7 and MagR-DiaBlo N=64 gets 44.1.

Cost on a T4: each 7B 4-bit training run takes many hours, spread over several sessions (it is resumable), and HF-generate evaluation takes about 3–6 hours (also resumable).
It needs approved access to `meta-llama/Llama-2-7b-hf` and an `HF_TOKEN` Colab secret.

In [ ]:
# ── Common setup (run every session) ─────────────────────────────────────────
REPO_URL = "https://github.com/<YOUR_GITHUB_USER>/DiaBlo.git"   # your fork with the colab-repro branch
BRANCH   = "colab-repro"
MODEL    = "Llama2-7B"                                       # short name or HF id
HF_ID    = "meta-llama/Llama-2-7b-hf"

import os, subprocess, torch
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/diablo"
os.makedirs(DRIVE, exist_ok=True)
DATA    = f"{DRIVE}/datasets/{MODEL}"     # tokenized data is tokenizer-specific
RESULTS = f"{DRIVE}/results"
EVALS   = f"{DRIVE}/eval"
os.environ["HF_HOME"] = f"{DRIVE}/hf_cache"   # keep downloads across sessions
os.environ["WANDB_MODE"] = "disabled"

REPO = "/content/DiaBlo"
if not os.path.exists(REPO):
    if os.path.exists(f"{DRIVE}/DiaBlo"):          # alternative: repo uploaded to Drive
        subprocess.run(["cp", "-r", f"{DRIVE}/DiaBlo", REPO], check=True)
    else:
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
%cd {REPO}
!git log -1 --oneline 2>/dev/null || true
!pip -q install peft
print(torch.__version__, torch.cuda.get_device_name(0), "bf16:", torch.cuda.is_bf16_supported())

In [ ]:
# Only needed for gated models (Llama). Add HF_TOKEN in Colab's "Secrets" (key icon) panel.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN set")
except Exception as e:
    print("No HF_TOKEN secret (fine for Qwen):", e)

In [ ]:
!pip -q install bitsandbytes
if not os.path.exists(f"{DATA}/GSM8k_processed/train"):
    !python data_processing/process_gsm8k.py --model_name {HF_ID} --output_dir {DATA} --max_length 512

In [ ]:
RUN_ID = "Q1"   # Q1 = NF4 DiaBlo N=64, Q2 = QLoRA r=64 (the paper's LoRA rank for this table)
RUNS = {"Q1": "--method diablo --num_blocks 64 --lr 5e-4",
        "Q2": "--method lora --lora_r 64 --lr 2e-4"}
CMD = (f"python train.py --model_name {MODEL} --task GSM8k --data_dir {DATA} --output_dir {RESULTS} --run_name {RUN_ID} --resume "
       f"--load_in_4bit --precision fp16 --gradient_checkpointing --per_device_batch_size 4 --gradient_accumulation_steps 4 "
       f"--max_epochs 3 --warmup_steps 100 --dropout 0.05 --target_modules qkvud --save_steps 50 {RUNS[RUN_ID]}")
print(CMD)

In [ ]:
!mkdir -p {RESULTS}/{RUN_ID}
!{CMD} 2>&1 | tee -a {RESULTS}/{RUN_ID}/train.log

In [ ]:
# Resumable eval (4-bit adapters can't be merged, so it uses HF generate). Re-run after a disconnect.
import json
cfg = json.load(open(f"{RESULTS}/{RUN_ID}/run_config.json"))
size = f"--num_blocks {cfg['num_blocks']}" if cfg["method"] == "diablo" else f"--lora_r {cfg['lora_r']}"
!python evaluate_gsm8k.py --model_name {MODEL} --method {cfg['method']} {size} --load_in_4bit --precision fp16 \
    --adapter_path {RESULTS}/{RUN_ID}/adapter.chkpt --data_dir {DATA} --batch_size 16 --output_file {EVALS}/{RUN_ID}_hf.jsonl